<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/projects/01-clothing-reviews/notebook.ipynb)


# Project 01 — What are customers really saying?

An online clothing shop has a thousand reviews and nobody with time to read them.
The support team wants two things: to know **what people keep talking about**, and,
when a new review arrives, to see **the reviews most like it** — so a reply can be
written by someone who has seen that kind of feedback before.

You will build both, with **a model running on your own machine** and a **vector
database**. No API key, no account, nothing sent anywhere.

## The data

`data/reviews.csv` — 1,000 real reviews, anonymised by their publisher and released
into the public domain (CC0). You need one column:

| Column | What it holds |
|---|---|
| `Review Text` | What the customer wrote about the product and the purchase |

## What you deliver

| Task | Store it in | What it is |
|---|---|---|
| 1 | `embeddings` | one vector per review |
| 2 | `embeddings_2d` | the same reviews as 2-D points, and a plot of them |
| 3 | `topic_reviews` | a dict: topic → the reviews closest to it |
| 4 | `most_similar_reviews` | the 3 reviews most like *"Absolutely wonderful - silky and sexy and comfortable"* |

Each task ends with a check cell. The checks are **not counted** toward your marks.

**In class you watch it run. After class, you run it yourself.** Every task is
already written, and each code cell says, step by step, what it does and why.
Run the cells in order, read the comments, and look at what each one prints.
Nothing here is marked, and there is nothing to write.

## Before you start

This project needs one model: **`nomic-embed-text`**, a 274 MB embedding model.
Everyone in the class uses the same one, so your results match the screen.
The next section shows how to get it, on your laptop or on Google Colab.

## Get the model — pick where it runs

| | On your laptop | On Google Colab |
|---|---|---|
| **Choose it when** | Ollama runs on your machine. **8 GB of RAM is enough** for this model. | Ollama does not install on your machine, or you have no laptop with you. |
| **Needs** | Ollama, and the course installed with `uv` | A Google account. Nothing on your laptop. |
| **Lasts** | For good | Until Colab gives you a new machine. Then run the Colab cell again (about 3 minutes). |

### On your laptop

1. Install Ollama from **[ollama.com/download](https://ollama.com/download)**.
2. In a terminal, from your course folder:

```bash
uv sync --extra projects          # ChromaDB, scikit-learn, pandas, matplotlib
ollama pull nomic-embed-text      # the model: 274 MB
ollama list                       # nomic-embed-text must be in the list
```

3. Open this notebook with `uv run jupyter lab`, **skip the Colab cell below**, and
   run the setup cell after it. It prints `ready: …`.

If the setup cell says *no Ollama server*, start one in a terminal with
`ollama serve` and run the cell again.

### On Google Colab

1. Click **Open in Colab** at the top of this notebook.
2. Optional, but faster: **Runtime → Change runtime type → T4 GPU → Save**. The
   model also runs on the free CPU; embedding the reviews then takes a few minutes
   instead of seconds.
3. Run the **Colab cell** below. It fetches the course, installs ChromaDB, installs
   Ollama **inside the Colab machine**, and pulls `nomic-embed-text`. About 3 minutes.
4. Run the setup cell after it. It prints `ready: …`.

Nothing leaves Colab: the model runs on the same machine as the notebook, so
`localhost` is correct there too. No key, no account other than Google's.

**Your work in Colab is not in your course folder.** These project checks are not
counted, so that is fine. To keep your code, use **File → Save a copy in Drive**.

In [ ]:
# manual-run: needs Ollama with nomic-embed-text (on your laptop, or set up by this cell on Colab)
# Google Colab only. On your laptop, this cell does nothing: skip it.
import sys

if "google.colab" not in sys.modules:
    print("Not on Colab — nothing to do here. Run the next cell.")
else:
    import os
    import shutil
    import subprocess
    import time
    import urllib.request
    from pathlib import Path

    COURSE = Path("/content/dev3pack-cohort-2026-09")
    OLLAMA_LOG = Path("/content/ollama.log")

    if not (COURSE / "pyproject.toml").exists():
        print("1/4 fetching the course…")
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1",
             "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(COURSE)],
            check=True,
        )
    os.chdir(COURSE)

    print("2/4 installing ChromaDB…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "chromadb>=1.0,<2", "python-dotenv"],
        check=True,
    )

    def ollama_up() -> bool:
        try:
            urllib.request.urlopen("http://localhost:11434", timeout=2)
            return True
        except OSError:
            return False

    if not ollama_up():
        if shutil.which("ollama") is None:
            print("3/4 installing Ollama in this Colab machine (about 30 seconds)…")
            # The installer unpacks a .tar.zst archive, and zstd is not always present.
            subprocess.run("apt-get -qq install -y zstd > /dev/null", shell=True, check=False)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["nohup", "ollama", "serve"], stdout=OLLAMA_LOG.open("w"),
                         stderr=subprocess.STDOUT)
        for _ in range(30):
            if ollama_up():
                break
            time.sleep(1)
        else:
            raise SystemExit(f"Ollama did not start. Read {OLLAMA_LOG}")

    print("4/4 pulling nomic-embed-text (274 MB)…")
    subprocess.run(["ollama", "pull", "nomic-embed-text"], check=True, capture_output=True)
    print("ready on Colab. Now run the setup cell below.")


In [ ]:
# manual-run: needs a local Ollama server, an embedding model, and the `projects` extra
import json
import sys
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import chromadb
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from scipy.spatial import distance
    from sklearn.manifold import TSNE
except ImportError as error:
    raise SystemExit(f"missing {error.name!r}. Run: uv sync --extra projects") from None

MODEL = "nomic-embed-text"
OLLAMA = "http://localhost:11434"

try:
    with urllib.request.urlopen(f"{OLLAMA}/api/tags", timeout=5) as response:
        pulled = {m["name"].split(":")[0] for m in json.loads(response.read())["models"]}
except (urllib.error.URLError, OSError):
    raise SystemExit("no Ollama server on localhost:11434. Start it with: ollama serve") from None
if MODEL not in pulled:
    raise SystemExit(f"the model is not pulled yet. Run: ollama pull {MODEL}")

from bootcamp_agent.bonus import bonus
from bootcamp_agent.projects import clothing_reviews  # noqa: F401  (registers the checks)

print(f"ready: chromadb {chromadb.__version__}, model {MODEL}")

## Look at the data before you trust it

Real data is never quite what the column name promises.

In [ ]:
reviews = pd.read_csv(ROOT / "projects/01-clothing-reviews/data/reviews.csv")
print(f"{len(reviews)} rows")
print(f"{reviews['Review Text'].isna().sum()} with no review text at all")
reviews[["Review Text", "Rating", "Class Name"]].head()

Some rows have **no text**. An empty review still gets an embedding if you ask for
one — a vector that means nothing, sitting among the real ones and turning up in
searches. Decide what to do with them before you embed anything.

---

## Task 1 — Create the embeddings

**Store in `embeddings`** · about 10 minutes · check: `project-01-e1`

An embedding turns text into a list of numbers, placed so that texts which *mean*
similar things sit close together. That closeness is what every later task uses.

**Done when** `embeddings` holds one vector per review that has text, and the check
is green.

The helper below talks to your local model. It is the equivalent of
`client.embeddings.create()` — you call it with a list of texts and get back a list
of vectors.

<details><summary><b>Hint</b></summary>

- Take the `Review Text` column and drop the empty ones first: `.dropna()`
- Sending 958 texts at once is slow and can time out. Send them in batches of 64.
- `embed(batch)` returns a list; extend `embeddings` with it.

</details>

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    """Embeddings for a batch of texts, from the model on your machine."""
    request = urllib.request.Request(
        f"{OLLAMA}/api/embed",
        data=json.dumps({"model": MODEL, "input": texts}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=300) as response:
        return json.loads(response.read())["embeddings"]


vector = embed(["Runs small, order a size up."])[0]
print(f"one review -> {len(vector)} numbers, starting {[round(x, 3) for x in vector[:4]]}")

In [ ]:
# Task 1 — create the embeddings.

# STEP 1. Take the review texts, and drop the rows with no text.
#         An empty review would still get a vector: a vector that means nothing,
#         and it would turn up in every search. So we remove it first.
review_texts = reviews["Review Text"].dropna().tolist()
print(f"{len(review_texts)} reviews with text")

# STEP 2. Turn every review into a vector (a list of 768 numbers).
#         We send 64 reviews per request. Sending all of them at once is slow,
#         and a very large request can time out.
embeddings = []
for start in range(0, len(review_texts), 64):
    batch = review_texts[start:start + 64]      # the next 64 reviews
    embeddings.extend(embed(batch))             # one vector per review, in the same order

# STEP 3. Check the result: one vector per review.
print(f"{len(embeddings)} embeddings, {len(embeddings[0])} numbers each")

bonus("project-01-e1", embeddings)


---

## Task 2 — Reduce to 2-D and look at it

**Store in `embeddings_2d`** · about 10 minutes · check: `project-01-e2`

Each embedding has hundreds of numbers, and nobody can look at 768 dimensions.
t-SNE squeezes them down to two while trying to keep neighbours as neighbours, so
you can *see* whether similar reviews really do sit together.

**Done when** `embeddings_2d` is one `(x, y)` point per review, you have a scatter
plot, and the check is green.

<details><summary><b>Hint</b></summary>

- `TSNE(n_components=2, random_state=0).fit_transform(np.array(embeddings))`
- `plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=6)`
- Colour the points by `Rating` and see whether the unhappy reviews cluster.

</details>

**Read the plot honestly.** t-SNE keeps *local* neighbourhoods. The distance between
two far-apart clusters means very little, so do not read one as "twice as different".

In [ ]:
# Task 2 — reduce to 2-D and look at it.

# STEP 1. Squeeze 768 numbers per review down to 2, so we can draw them.
#         t-SNE tries to keep neighbours as neighbours. random_state=0 makes the
#         picture the same every time, so your plot matches the screen.
embeddings_2d = TSNE(n_components=2, random_state=0).fit_transform(np.array(embeddings))

# STEP 2. Get each review's star rating, in the same order as review_texts.
#         We dropped the empty reviews in Task 1, so we drop them here too.
ratings = reviews.loc[reviews["Review Text"].notna(), "Rating"].to_numpy()

# STEP 3. Draw one dot per review, coloured by rating: red = 1 star, green = 5 stars.
plt.figure(figsize=(8, 6))
points = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=ratings, cmap="RdYlGn", s=8)
plt.colorbar(points, label="rating")
plt.title(f"{len(review_texts)} reviews, placed by meaning")
plt.axis("off")   # the x and y values mean nothing on their own
plt.show()

# What to look for: do dots of the same colour sit together?
# Remember: close dots are similar. Far-apart groups are NOT "twice as different".

bonus("project-01-e2", embeddings_2d)


---

## Task 3 — What do people keep talking about?

**Store in `topic_reviews`** · about 15 minutes · check: `project-01-e3`

Embed a few **topic words** — `quality`, `fit`, `style`, `comfort` — the same way
you embedded the reviews. Then, for each topic, find the reviews whose embeddings
are closest to it.

**Done when** `topic_reviews` maps at least three topics to real reviews, and the
check is green.

<details><summary><b>Hint</b></summary>

- `embed(["quality", "fit", "style", "comfort"])` gives one vector per topic.
- `distance.cosine(a, b)` is **0** for identical direction and grows as they differ.
- Sort the reviews by their distance to a topic and keep the closest three.

</details>

**Now read what came back.** Does every review under `quality` actually talk about
quality? Write down how many of the three do. You will need that number.

In [ ]:
# Task 3 — what do people keep talking about?

# STEP 1. Embed the topic words, the same way we embedded the reviews.
#         A topic is just more text, so it gets a vector too.
topics = ["quality", "fit", "style", "comfort"]
topic_vectors = embed(topics)          # one vector per topic, in the same order

# STEP 2. For each topic, sort ALL reviews by their distance to the topic,
#         and keep the 3 closest.
#         distance.cosine is 0 when two vectors point the same way, and grows
#         as they point apart. Smaller = more similar.
topic_reviews = {}
for topic, topic_vector in zip(topics, topic_vectors):
    closest = sorted(
        range(len(review_texts)),                                     # every review's position
        key=lambda i: distance.cosine(embeddings[i], topic_vector),   # sort by distance to the topic
    )[:3]                                                             # keep the 3 closest
    topic_reviews[topic] = [review_texts[i] for i in closest]

# STEP 3. Print the start of each review we found.
#         Read them: does each one really talk about its topic?
for topic, found in topic_reviews.items():
    print(f"\n[{topic}]")
    for text in found:
        print("  -", text[:95])

bonus("project-01-e3", topic_reviews)


### Stand above it — read the whole review before you judge

Look at what came back under `quality`. If you only read the first line of each
review, some of them seem to be about colour, or styling — and it is tempting to
decide the search is bad.

**Read them in full before you decide.** The cell below prints where the word
actually appears in each one.

In [ ]:
if not topic_reviews.get("quality"):
    raise SystemExit("Finish Task 3 first: this reads what topic_reviews found for 'quality'.")

for review in topic_reviews["quality"]:
    at = review.lower().find("quality")
    opening = review[:60].replace("\n", " ")
    if at < 0:
        print(f"- {opening}...\n    (never says 'quality')\n")
    else:
        print(f"- {opening}...")
        print(f"    ...{review[max(0, at - 50):at + 45]}...\n")

Some of those reviews do not *open* with quality — they are **complaints** about it,
further in. The search was right. A glance at the first line would have said it
was wrong.

### And measure a change instead of assuming it

This model's documentation says to put `search_document: ` before the texts you
store and `search_query: ` before what you search with. Documentation says to do
it, so it must help — right? Measure it.

In [ ]:
if len(embeddings) != len(review_texts) or not review_texts:
    raise SystemExit("Finish Task 1 first: this compares against your embeddings.")


def top3(query_vector, document_vectors):
    order = sorted(range(len(review_texts)), key=lambda i: distance.cosine(document_vectors[i], query_vector))
    return [review_texts[i] for i in order[:3]]


prefixed = []
for start in range(0, len(review_texts), 64):
    prefixed.extend(embed(["search_document: " + t for t in review_texts[start:start + 64]]))

for label, hits in (
    ("without prefixes", top3(embed(["quality"])[0], embeddings)),
    ("with prefixes", top3(embed(["search_query: quality"])[0], prefixed)),
):
    mentions = sum("quality" in text.lower() for text in hits)
    print(f"{label}: {mentions} of 3 talk about quality")
    for text in hits:
        at = text.lower().find("quality")
        print("   -", text[max(0, at - 30):at + 50].replace("\n", " "))
    print()

On this data, for this topic, **both find reviews about quality.** What changed is
*which* ones: without prefixes you get complaints and praise mixed; with them,
mostly praise.

That is not "better" or "worse" until you say what the support team needs. If the
point is to find unhappy customers, the version that surfaced complaints was more
useful — and the documented setting would have hidden them.

Two habits worth more than any setting: **read the whole result**, and **measure
before you claim an improvement**.

---

## Task 4 — Find the reviews most like this one

**Store in `most_similar_reviews`** · about 15 minutes · check: `project-01-e4`

Searching 958 vectors one by one is fine. Searching a million is not, and that is
what a **vector database** is for: it stores the embeddings once and answers
*"what is nearest to this?"* quickly. You will use ChromaDB.

Write a function that returns the **3 reviews most similar** to a given review, and
apply it to:

> *Absolutely wonderful - silky and sexy and comfortable*

**Done when** `most_similar_reviews` is a list of three review texts and the check
is green.

The class below plugs your local model into ChromaDB, so ChromaDB can embed text
itself. It uses the prefixes the model's documentation recommends — you have just
seen that they change *which* neighbours come back, so keep that in mind when you
read the results.

<details><summary><b>Hint</b></summary>

- `client = chromadb.Client()` — an in-memory database, gone when the notebook stops
- `client.create_collection(name=..., embedding_function=LocalEmbeddings())`
- `collection.add(documents=..., ids=...)` — ids are strings, one per review
- `collection.query(query_texts=[...], n_results=...)`

</details>

**Look closely at the first thing it returns.** Then decide whether that is what the
support team asked for.

In [ ]:
from chromadb import Documents, EmbeddingFunction, Embeddings


class LocalEmbeddings(EmbeddingFunction):
    """ChromaDB's seam for 'turn text into vectors', pointed at the local model."""

    def __init__(self) -> None:
        pass

    def __call__(self, input: Documents) -> Embeddings:
        return embed(["search_document: " + text for text in input])

    def embed_query(self, input: Documents) -> Embeddings:
        return embed(["search_query: " + text for text in input])

    @staticmethod
    def name() -> str:
        return "local-nomic-embed-text"

    def get_config(self) -> dict:
        return {}

    @staticmethod
    def build_from_config(config: dict) -> "LocalEmbeddings":
        return LocalEmbeddings()


print("embedding function ready")

In [ ]:
# Task 4 — find the reviews most like this one.

# STEP 1. Start an in-memory vector database. It disappears when the notebook stops.
client = chromadb.Client()

# STEP 2. Create a collection: a table of reviews and their vectors.
#         embedding_function: ChromaDB calls our local model to make the vectors.
#         "hnsw:space": "cosine": measure closeness the same way as in Task 3.
collection = client.create_collection(
    name="reviews",
    embedding_function=LocalEmbeddings(),
    metadata={"hnsw:space": "cosine"},
)

# STEP 3. Add every review, 128 at a time. Each one needs a unique id, as a string.
#         ChromaDB embeds the text itself, using LocalEmbeddings.
for start in range(0, len(review_texts), 128):
    batch = review_texts[start:start + 128]
    collection.add(documents=batch, ids=[str(start + i) for i in range(len(batch))])
print(f"{collection.count()} reviews stored")


# STEP 4. The search. We ask for n + 1 results, because the review we search with
#         is in the database too, and the nearest review to itself is itself.
#         We remove it, and keep n.
def find_similar_reviews(text: str, n: int = 3) -> list[str]:
    """The n reviews most similar to `text` -- never `text` itself."""
    found = collection.query(query_texts=[text], n_results=n + 1)["documents"][0]
    return [review for review in found if review != text][:n]


# STEP 5. Prove the trap from STEP 4 is real: search with a review that is in the data.
query = "Absolutely wonderful - silky and sexy and comfortable"
first = collection.query(query_texts=[query], n_results=1)["documents"][0][0]
print(f"is the nearest review the query itself? {first == query}\n")

# STEP 6. The answer the support team asked for: the 3 most similar OTHER reviews.
most_similar_reviews = find_similar_reviews(query)
for review in most_similar_reviews:
    print("-", review[:100])

bonus("project-01-e4", most_similar_reviews)


---

## What you just built

- **Embeddings** turn meaning into position. Close means similar.
- **Dimensionality reduction** makes that position something you can look at — for
  neighbourhoods, not for distances between far-apart groups.
- **Topics** are just more embeddings. Search for them the way you search for anything.
- **Read the whole result, and measure a change** — a first-line glance and a
  documented setting both told a story the data did not support.
- **A vector database** stores the vectors once and answers "what is nearest" fast —
  and the nearest thing to a review is always **that review**.

## Stuck? Ask the course

The coach answers from the course pages, offline, with no key.

In [ ]:
from bootcamp_agent.coach import coach

coach("embeddings cosine distance vector database nearest neighbour", top_k=1, max_chars=700)

## Your turn

Three more questions about the same reviews. The code is written: run each cell,
read the comments, and then change the text in quotes and run it again. Nothing
here is checked.

### 1. Replace the word with a sentence

Task 3 searched with one word: `quality`. A real support agent asks longer
questions. Does a sentence find better reviews?

In [ ]:
# STEP 1. The two searches we compare: one word, and one sentence that says what we mean.
WORD = "quality"
SENTENCE = "reviews about fabric quality and how well the clothes are made"

# STEP 2. Search the review vectors from Task 1 with each one.
#         closest() sorts every review by its distance to the search, and keeps the 3 closest.
def closest(search_text: str, how_many: int = 3) -> list[str]:
    search_vector = embed([search_text])[0]
    scored = []
    for text, vector in zip(review_texts, embeddings):
        scored.append((distance.cosine(vector, search_vector), text))
    scored.sort()                                  # smallest distance first
    return [text for _, text in scored[:how_many]]


# STEP 3. For each result, show which quality words it contains.
#         A word count is a rough guide, not proof: a review can say "made" and be
#         about something else. Read the review before you trust the count.
ABOUT_IT = ("quality", "fabric", "material", "made", "cheap", "thin", "pilling")

for label, search_text in (("one word", WORD), ("a sentence", SENTENCE)):
    print(f"{label}: {search_text!r}")
    for text in closest(search_text):
        words = [word for word in ABOUT_IT if word in text.lower()]
        print(f"   - {text[:90]}")
        print(f"     quality words: {words or 'none'}")
    print()

# Try it: change SENTENCE to your own question, and run the cell again.


### 2. Find the unhappy customers

Do the 1-star and 2-star reviews sit together on the map from Task 2? And what
does the review nearest to an unhappy one say?

In [ ]:
# STEP 1. Mark the unhappy reviews: rating 1 or 2.
#         `ratings` comes from Task 2, in the same order as review_texts.
unhappy = ratings <= 2
print(f"{unhappy.sum()} unhappy reviews out of {len(ratings)}")

# STEP 2. Draw the same map as Task 2: every review in grey, the unhappy ones in red.
plt.figure(figsize=(8, 6))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c="lightgrey", s=6)
plt.scatter(embeddings_2d[unhappy, 0], embeddings_2d[unhappy, 1], c="red", s=12, label="rating 1 or 2")
plt.legend()
plt.title("Where the unhappy customers are")
plt.axis("off")
plt.show()

# STEP 3. Take the first unhappy review, and find the 3 reviews closest to it.
#         We ask for 4 and skip the first one, because the closest review to a review is itself.
first_unhappy = review_texts[int(unhappy.argmax())]
print("An unhappy review:\n   ", first_unhappy[:150], "\n")
print("The reviews closest to it:")
for text in closest(first_unhappy, how_many=4)[1:]:
    print("   -", text[:120])

# What to look for: are the red dots in one place, or everywhere?
# Do the closest reviews complain about the same thing?


### 3. Answer a brand-new review

A new review arrives. It is not in the data. Which old reviews look like it? Those
are the ones a support agent should read before replying.

In [ ]:
# STEP 1. Write a new review. Change this text to anything you like.
NEW_REVIEW = "The dress looked great online but the zipper broke after one wear."

# STEP 2. Ask the vector database from Task 4 for the most similar reviews.
#         find_similar_reviews() embeds the text, searches, and returns 3 reviews.
similar = find_similar_reviews(NEW_REVIEW)

# STEP 3. Show what the support agent would read.
print("New review:\n   ", NEW_REVIEW, "\n")
print("Read these first:")
for text in similar:
    print("   -", text[:140])

# Try it: write a happy review, then an angry one. Do the results change the way you expect?
